In [9]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [10]:
sim = BasicSimulator()   # simulator backend used throughout

N               = 100   # number of qubits Alice transmits
CHECK_FRACTION  = 0.5   # fraction of sifted key sacrificed for error checking
ERROR_THRESHOLD = 0.05  # abort if error rate exceeds 5%

In [11]:
def quantum_random_bits(n):
    bits = []
    while len(bits) < n:
        size = min(20, n - len(bits))
        qc = QuantumCircuit(size, size)
        for i in range(size):
            qc.h(i)                           # |0> --> (|0>+|1>) / sqrt(2)
        qc.measure(range(size), range(size))  # collapses to 0 or 1 at random
        job = sim.run(transpile(qc, sim), shots=1, memory=True)
        bits_str = job.result().get_memory()[0]
        bits.extend([int(b) for b in reversed(bits_str)])
    return bits[:n]

# Quick demonstration — show a quantum random number generator circuit
demo_qc = QuantumCircuit(4, 4)
for i in range(4):
    demo_qc.h(i)
demo_qc.measure(range(4), range(4))
print("QRNG circuit (4 bits):")
print(demo_qc.draw())
print("Sample output:", quantum_random_bits(8))

QRNG circuit (4 bits):
     ┌───┐┌─┐         
q_0: ┤ H ├┤M├─────────
     ├───┤└╥┘┌─┐      
q_1: ┤ H ├─╫─┤M├──────
     ├───┤ ║ └╥┘┌─┐   
q_2: ┤ H ├─╫──╫─┤M├───
     ├───┤ ║  ║ └╥┘┌─┐
q_3: ┤ H ├─╫──╫──╫─┤M├
     └───┘ ║  ║  ║ └╥┘
c: 4/══════╩══╩══╩══╩═
           0  1  2  3 
Sample output: [1, 1, 0, 0, 0, 0, 1, 1]


In [12]:
# ── ALICE ──────────────────────────────────────────────────────────────────

def alice_encode(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)      # flip |0> to |1>
    if basis == 1:
        qc.h(0)      # rotate to X-basis
    return qc

alice_bits  = quantum_random_bits(N)  # Alice's secret key bits
alice_bases = quantum_random_bits(N)  # 0 = Z-basis, 1 = X-basis

# Alice prepares one qubit per bit — these travel to Bob over the quantum channel
transmitted_qubits = [alice_encode(alice_bits[i], alice_bases[i]) for i in range(N)]

print("[ALICE] Key bits (first 20) :", alice_bits[:20])
print("[ALICE] Bases   (first 20) :", alice_bases[:20])
print(f"[ALICE] {N} qubits prepared and sent to Bob.")

# Show all four possible encoding circuits
print("\nEncoding circuits:")
for bit in [0, 1]:
    for basis in [0, 1]:
        label = {(0,0):"|0>", (1,0):"|1>", (0,1):"|+>", (1,1):"|->"}
        print(f"  bit={bit}, basis={'Z' if basis==0 else 'X'} --> {label[(bit,basis)]}")
        print(alice_encode(bit, basis).draw())

[ALICE] Key bits (first 20) : [0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0]
[ALICE] Bases   (first 20) : [1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0]
[ALICE] 100 qubits prepared and sent to Bob.

Encoding circuits:
  bit=0, basis=Z --> |0>
     
  q: 
     
c: 1/
     
  bit=0, basis=X --> |+>
     ┌───┐
  q: ┤ H ├
     └───┘
c: 1/═════
          
  bit=1, basis=Z --> |1>
     ┌───┐
  q: ┤ X ├
     └───┘
c: 1/═════
          
  bit=1, basis=X --> |->
     ┌───┐┌───┐
  q: ┤ X ├┤ H ├
     └───┘└───┘
c: 1/══════════
               


In [13]:
# ── BOB ────────────────────────────────────────────────────────────────────

def bob_measure(qubit_qc, basis):
    qc = qubit_qc.copy()
    if basis == 1:
        qc.h(0)       # rotate X-basis back to Z-basis before measuring
    qc.measure(0, 0)
    job = sim.run(transpile(qc, sim), shots=1, memory=True)
    return int(job.result().get_memory()[0])

bob_bases   = quantum_random_bits(N)   # Bob's random measurement bases (0=Z, 1=X)
bob_results = [bob_measure(transmitted_qubits[i], bob_bases[i]) for i in range(N)]

print("[BOB] Bases   (first 20):", bob_bases[:20])
print("[BOB] Results (first 20):", bob_results[:20])

[BOB] Bases   (first 20): [1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0]
[BOB] Results (first 20): [0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0]


In [14]:
# ── SIFTING (public classical channel) ─────────────────────────────────────

sifted_alice = []
sifted_bob   = []

for i in range(N):
    if alice_bases[i] == bob_bases[i]:      # same basis → reliable bit
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])

print(f"Transmitted   : {N} qubits")
print(f"After sifting : {len(sifted_alice)} bits  (~{len(sifted_alice)/N*100:.0f}%, expected ~50%)")
print(f"\nSifted key (Alice) : {sifted_alice[:20]} ...")
print(f"Sifted key (Bob)   : {sifted_bob[:20]} ...")

Transmitted   : 100 qubits
After sifting : 49 bits  (~49%, expected ~50%)

Sifted key (Alice) : [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1] ...
Sifted key (Bob)   : [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1] ...


In [15]:
# ── ERROR CHECKING ─────────────────────────────────────────────────────────

n_sifted = len(sifted_alice)
n_check  = math.ceil(n_sifted * CHECK_FRACTION)

# Select check positions using quantum randomness:
# assign each sifted position a random 4-bit priority, sort, take the lowest n_check.
rand_bits  = quantum_random_bits(n_sifted * 4)
priorities = [sum(rand_bits[i*4 + b] << b for b in range(4)) for i in range(n_sifted)]
all_pos    = sorted(range(n_sifted), key=lambda i: priorities[i])
check_idx  = sorted(all_pos[:n_check])
key_idx    = [i for i in range(n_sifted) if i not in check_idx]

check_alice = [sifted_alice[i] for i in check_idx]
check_bob   = [sifted_bob[i]   for i in check_idx]

errors     = sum(a != b for a, b in zip(check_alice, check_bob))
error_rate = errors / n_check

print(f"Sifted key length  : {n_sifted}")
print(f"Bits used for check: {n_check}")
print(f"Errors found       : {errors}")
print(f"Error rate         : {error_rate*100:.1f}%  (threshold: {ERROR_THRESHOLD*100:.0f}%)")
print()

if error_rate > ERROR_THRESHOLD:
    print("ERROR RATE EXCEEDS THRESHOLD — possible eavesdropping! Protocol aborted.")
else:
    print("No errors detected. No eavesdropping. Protocol continues.")

Sifted key length  : 49
Bits used for check: 25
Errors found       : 0
Error rate         : 0.0%  (threshold: 5%)

No errors detected. No eavesdropping. Protocol continues.


In [16]:
# ── FINAL KEY ──────────────────────────────────────────────────────────────

final_key_alice = [sifted_alice[i] for i in key_idx]
final_key_bob   = [sifted_bob[i]   for i in key_idx]

print(f"Final key length : {len(final_key_alice)} bits")
print(f"Alice's key      : {final_key_alice}")
print(f"Bob's key        : {final_key_bob}")
print()

if final_key_alice == final_key_bob:
    print("Keys match — Alice and Bob share an identical secret key.")
else:
    print("Keys do not match.")

Final key length : 24 bits
Alice's key      : [0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1]
Bob's key        : [0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1]

Keys match — Alice and Bob share an identical secret key.
